# LLM-as-judge

**Session 3 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Grade open-ended outputs with a model, using a rubric and structured output. The lesson is
**a judge is itself a classifier you must validate** — measure its agreement with your own
labels on deliberately borderline cases. It will not be 100%. A weak judge model is worse.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import json
from utils import ask, SMALL_MODEL, BIG_MODEL
from eval import load_cases


In [ ]:
RUBRIC = """You grade whether an ANSWER is faithful to a CONTEXT. Faithful means every claim in
the ANSWER is supported by the CONTEXT. Reasonable paraphrase and rounding are fine.
Adding facts not in the CONTEXT, contradicting it, or saying "I don't know" when the CONTEXT
does contain the answer are all unfaithful.

Score:
  5 = fully supported, no added claims
  4 = supported, minor harmless imprecision
  3 = mostly supported but one unsupported detail
  2 = a claim contradicts or is absent from the context
  1 = largely unsupported or refuses despite the context having the answer

Return ONLY JSON: {"score": <1-5>, "reason": "<one sentence>"}"""

def judge(answer, context, model=SMALL_MODEL):
    raw = ask(f"{RUBRIC}\n\nCONTEXT:\n{context}\n\nANSWER:\n{answer}", model=model)
    try:
        obj = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
        return {"score": int(obj["score"]), "reason": obj.get("reason", "")}
    except (json.JSONDecodeError, ValueError, KeyError):
        return {"score": None, "reason": "unparseable: " + raw[:80]}

print(judge("Paris is the capital.", "France is a country in Europe. Its capital is Paris."))

### Validate the judge against your own labels

`eval/datasets/judge_faithfulness.jsonl` has 18 (context, answer, human) rows — several are
deliberately borderline (paraphrase, rounding, partial answers). Treat score ≥ 4 as "pass"
and compare to the human label. Report the agreement rate and list every disagreement.

In [ ]:
LABELLED = load_cases("../eval/datasets/judge_faithfulness.jsonl")

def validate(model):
    agree, disagreements = 0, []
    for row in LABELLED:
        v = judge(row["answer"], row["context"], model=model)
        verdict = "pass" if (v["score"] or 0) >= 4 else "fail"
        if verdict == row["human"]:
            agree += 1
        else:
            disagreements.append((row, v, verdict))
    print(f"{model}: agrees with human on {agree}/{len(LABELLED)}")
    for row, v, verdict in disagreements:
        print(f"  judge={verdict}(score {v['score']}) human={row['human']} | "
              f"ctx={row['context'][:45]!r} ans={row['answer'][:45]!r}")
    return agree / len(LABELLED)

small_agreement = validate(SMALL_MODEL)

### Does a bigger judge model agree with you more?

The judge is the last thing between you and a bad eval. Run the identical validation on the
big model and compare agreement.

In [ ]:
try:
    big_agreement = validate(BIG_MODEL)
    print(f"\nagreement:  {SMALL_MODEL} {small_agreement:.0%}   vs   {BIG_MODEL} {big_agreement:.0%}")
except Exception as e:
    print(f"skipped big-model judge: {type(e).__name__}: {e}")

# Only trust judge_scorer for run_eval if agreement is high enough.
def judge_scorer(output, expected):
    """expected is the reference context; pass = faithful (score >= 4)."""
    return (judge(output, expected).get("score") or 0) >= 4


## Your turn - vary the example

1. Add 3 borderline answers where you and the judge might disagree; check agreement.
2. Tighten the rubric in `judge()` (define what a 3 vs a 4 means) and re-check.
3. Run `run_eval(cases, my_answer_fn, scorer=judge_scorer)` on a small set.
